# QC структурных построений: кросс-плот и кросс-валидация (МОГТ 2D/3D)

Валидация точности структурной карты против истинных глубин геологических маркеров в скважинах.

**План работы:**
1. Базовые метрики соответствия существующей карты скважинам (без CV)
2. Диагностика: невязка от глубины, карта невязок в плане
3. Честная кросс-валидация суррогатной поверхности `Z_well ~ f(X, Y)` четырьмя схемами:
   `Random K-Fold` (некорректно) → `Leave-One-Well-Out` → `Block CV` → `Stratified by Zone` (лучший вариант)
4. Сравнение схем CV и разбор выбросов

Автор: Алексей · геофизик → data scientist


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import structural_qc_full as sq

mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.grid": True,
    "grid.color": "#E5E5E5",
    "font.size": 11,
})
RED, BLUE = "#C23A22", "#2F5FA8"

## 1. Загрузка данных

Ожидаемый формат: `well_id, X, Y, Z_well, Z_map`, опционально `horizon_id`, `tectonic_zone`, `quality_flag`.

In [ ]:
df = sq.load_and_validate("data/structural_wells_100.csv")
print(f"n = {len(df)} скважин")
df.head()

In [ ]:
df['quality_flag'].unique()

In [ ]:
df.describe()

In [ ]:
df.groupby("tectonic_zone")["Z_well"].agg(["count", "mean", "std"]).round(1)

## 2. Базовый кросс-плот (без кросс-валидации)

Z_well (X) против Z_map (Y) — насколько существующая карта уже сходится со скважинами.

In [ ]:
basic = sq.basic_crossplot_stats(df)
print(f"R²      = {basic.r2:.3f}")
print(f"slope   = {basic.slope:.3f}   intercept = {basic.intercept:+.2f} м")
print(f"RMSE    = {basic.rmse:.2f} м")
print(f"MAE     = {basic.mae:.2f} м")
print(f"Bias    = {basic.bias:+.2f} м")
print(f"Std     = {basic.std:.2f} м")
print(f"Rel.err = {basic.rel_error_pct:.2f} %")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

lo, hi = df[["Z_well","Z_map"]].min().min(), df[["Z_well","Z_map"]].max().max()
axes[0].plot([lo, hi], [lo, hi], "--", color="#999", label="Y = X")
axes[0].scatter(df["Z_well"], df["Z_map"], c=BLUE, edgecolor="#1F3F70", s=45, alpha=.85)
axes[0].set_xlabel("Z_well, м"); axes[0].set_ylabel("Z_map, м")
axes[0].set_title("Кросс-плот: карта vs скважины")
axes[0].legend()

axes[1].axhline(0, ls=":", color="#999")
axes[1].scatter(df["Z_well"], basic.residuals, c=RED, s=45, alpha=.85)
axes[1].set_xlabel("Z_well, м"); axes[1].set_ylabel("ΔZ = Z_map − Z_well, м")
axes[1].set_title("Невязка от глубины (проверка на тренд)")

plt.tight_layout()
plt.show()

**Наблюдение:** невязка растёт с глубиной — характерный признак ошибки скоростной модели
при глубинном преобразовании, а не случайного шума. Это стоит проверить отдельно от общего RMSE.

## 3. Карта невязок в плане

Зональность цвета укажет на систематическое смещение по площади (тектонический блок, качество сейсмики).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
vmax = np.abs(basic.residuals).max()
sc = ax.scatter(df["X"], df["Y"], c=basic.residuals, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, s=90, edgecolor="black", linewidth=.6)
plt.colorbar(sc, label="ΔZ, м")
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_aspect("equal")
ax.set_title("Карта невязок")
plt.tight_layout()
plt.show()

## 4. Честная кросс-валидация

Строим суррогатную поверхность `Z_well ~ f(X, Y)` (полином 1-й или 2-й степени по X,Y) и
валидируем её **процесс построения**, а не готовую карту — это иллюстрация того, как
следовало бы кросс-валидировать сам метод интерполяции/построения структурной карты,
если бы он использовал скважины в калибровке.

Сравниваем 4 схемы согласно плану — от заведомо оптимистичной до наиболее строгой.

In [ ]:
results, comparison = sq.run_all_cv(df, degree=1, k=5, grid_n=3)
comparison.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
colors = {"Random K-Fold": "#999999", "Leave-One-Well-Out": BLUE,
          "Block CV": "#5C8A7A", "Stratified by Zone": RED}
bars = ax.bar(comparison.index, comparison["rmse"],
              color=[colors.get(i, BLUE) for i in comparison.index])
for b, v in zip(bars, comparison["rmse"]):
    ax.text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.1f}", ha="center")
ax.set_ylabel("RMSE, м")
ax.set_title("RMSE по схемам CV — иллюзия точности Random K-Fold")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

gap = comparison.loc["Random K-Fold","rmse"]
worst = comparison.drop(index="Random K-Fold")["rmse"].max()
print(f"Random K-Fold RMSE = {gap:.1f} м, самая строгая схема = {worst:.1f} м "
      f"(разница {worst-gap:.1f} м — мера оптимизма случайного разбиения)")

## 5. Влияние степени тренд-поверхности

Проверим устойчивость результата к выбору степени полинома (1 — плоскость, 2 — квадратичная).

In [ ]:
for deg in [1, 2]:
    _, comp = sq.run_all_cv(df, degree=deg, k=5, grid_n=3)
    print(f"--- degree={deg} ---")
    print(comp[["rmse","mae","bias"]].round(2), "\n")

**Вывод:** более сложная (квадратичная) поверхность не улучшает, а иногда ухудшает
CV-метрики при таком количестве скважин и площади — признак переобучения тренда на
разреженном скважинном контроле. Простая плоскость надёжнее.

## 6. Разбор по фолдам (Stratified by Zone)

Посмотрим, есть ли зона, которая систематически хуже описывается.

In [ ]:
results["Stratified by Zone"]["folds"]

In [ ]:
fig, ax = plt.subplots(figsize=(6.5,6))
preds = results["Stratified by Zone"]["preds"]
zones = df["tectonic_zone"]
for zone, color in zip(zones.unique(), ["#4C72B0","#C23A22","#5C8A7A"]):
    m = zones == zone
    ax.scatter(df.loc[m,"Z_well"], preds[m], label=zone, s=55, alpha=.85, color=color)
lo, hi = df["Z_well"].min(), df["Z_well"].max()
ax.plot([lo,hi],[lo,hi],"--",color="#999")
ax.set_xlabel("Z_well, м"); ax.set_ylabel("Предсказано (CV), м")
ax.legend(); ax.set_title("Предсказано vs факт, по тектоническим зонам")
plt.tight_layout()
plt.show()

**Наблюдение:** зона `Central_Graben` даёт систематически больший разброс — соответствует
тектонически осложнённому блоку с более сложной сейсмической картиной (в синтетических
данных это заложено намеренно, в реальных данных так проявляется зона разломов/потери
когерентности).

## 7. Выбросы (LOWO, |невязка| > 2σ)

In [ ]:
resid_lowo = results["Leave-One-Well-Out"]["preds"] - df["Z_well"].to_numpy()
sd = resid_lowo.std(ddof=1)
mask = np.abs(resid_lowo) > 2*sd
out = df.loc[mask, ["well_id","X","Y","Z_well","tectonic_zone"]].copy()
out["Predicted_LOWO"] = results["Leave-One-Well-Out"]["preds"][mask]
out["Residual"] = resid_lowo[mask]
out

## 8. Итоги

- Существующая карта (`Z_map`) сходится со скважинами с RMSE ≈ первые метры–десятки метров
  (см. раздел 2) — но это **in-sample** оценка, не учитывающая возможную калибровку карты по
  тем же скважинам.
- Честная кросс-валидация суррогатной поверхности показывает существенно бо́льшую ошибку —
  ожидаемо, так как чисто скважинная интерполяция не воспроизводит детальную структуру,
  видимую в сейсмике.
- **Random K-Fold систематически оптимистичнее** пространственных схем — не должен
  использоваться для пространственно коррелированных геологических данных.
- Zone-stratified CV выявила проблемную тектоническую зону — точка для доработки
  скоростной модели/интерпретации разломов, а не просто "шум".

Тот же движок анализа лежит в основе интерактивного `app.py` (Streamlit).


In [ ]:
from streamlit_plotly_events import plotly_events